# Decision Tree Classifier (BoW)

## Config Setup

### 1. Import Libraries and setup hyperparameters

In [1]:
import sys
sys.path.append('..')

import numpy as np
import json
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score

from src.dataloader import DataLoader
from models.decision_trees.decision_tree import DecisionTreeModel
from helpers.save_model_params import extract_tree_to_dict

# Configs
json_path = '../models/decision_trees/fitted_bow_decision_tree.json'
train_data_path = '../src/data/train_data.csv'
val_data_path = '../src/data/validation_data.csv'
test_data_path = '../src/data/test_data.csv'
random_seed = 42

hyperparams = {
    "random_state": random_seed,
    "max_depth": [5, 10, 15, 20, 25, 30],
    "min_samples_split": [2, 4, 8, 16],
    "min_samples_leaf": [1, 5, 10],
    "criterion": ["gini", "entropy"],
}

### 2. Generate Training and Validation Data (BoW)

In [2]:
dataloader = DataLoader(random_seed)
X_train, y_train = dataloader.generate_Xt(train_data_path)
X_val, y_val = dataloader.generate_Xt(val_data_path)
X_test, y_test = dataloader.generate_Xt(test_data_path)

### 3. Generate fitted DTs w/ hyperparameter permutations to find best hyperparameters

In [5]:
def search_hyperparameters(X_train, y_train, X_val, y_val):
    best_acc = 0.0
    best_params = {}

    for max_depth in hyperparams["max_depth"]:
        for min_samples_split in hyperparams["min_samples_split"]:
            for min_samples_leaf in hyperparams["min_samples_leaf"]:
                for criterion in hyperparams["criterion"]:
                    model = DecisionTreeClassifier(
                        max_depth=max_depth,
                        min_samples_split=min_samples_split,
                        min_samples_leaf=min_samples_leaf,
                        criterion=criterion,
                        random_state=random_seed
                    )
                    model.fit(X_train, y_train)
                    y_val_pred = model.predict(X_val)
                    val_accuracy = accuracy_score(y_val, y_val_pred)

                    if val_accuracy > best_acc:
                        best_acc = val_accuracy
                        best_params = {
                            "max_depth": max_depth,
                            "min_samples_split": min_samples_split,
                            "min_samples_leaf": min_samples_leaf,
                            "criterion": criterion
                        }

                    print(f"Params: max_depth={max_depth}, min_samples_split={min_samples_split}, "
                            f"criterion={criterion}, min_samples_leaf={min_samples_leaf}, \n\t => Validation Accuracy: {val_accuracy:.4f}")

    return best_params

## Find Best DT

### 1. Find best hyperparameter permutations 

In [6]:
best_hyperparams = search_hyperparameters(X_train, y_train, X_val, y_val)
print("Best Hyperparameters:")
for param, value in best_hyperparams.items():
    print(f"{param}: {value}")

Params: max_depth=5, min_samples_split=2, criterion=gini, min_samples_leaf=1, 
	 => Validation Accuracy: 0.6267
Params: max_depth=5, min_samples_split=2, criterion=entropy, min_samples_leaf=1, 
	 => Validation Accuracy: 0.6133
Params: max_depth=5, min_samples_split=2, criterion=gini, min_samples_leaf=5, 
	 => Validation Accuracy: 0.6000
Params: max_depth=5, min_samples_split=2, criterion=entropy, min_samples_leaf=5, 
	 => Validation Accuracy: 0.5867
Params: max_depth=5, min_samples_split=2, criterion=gini, min_samples_leaf=10, 
	 => Validation Accuracy: 0.6200
Params: max_depth=5, min_samples_split=2, criterion=entropy, min_samples_leaf=10, 
	 => Validation Accuracy: 0.5933
Params: max_depth=5, min_samples_split=4, criterion=gini, min_samples_leaf=1, 
	 => Validation Accuracy: 0.6267
Params: max_depth=5, min_samples_split=4, criterion=entropy, min_samples_leaf=1, 
	 => Validation Accuracy: 0.6133
Params: max_depth=5, min_samples_split=4, criterion=gini, min_samples_leaf=5, 
	 => Valida

## Generate Best DT w/ Best Hyperparameters and Save Splits to JSON

### 1. Generate DT w/ best hyperparameters

In [4]:
best_hyperparams = {
    "max_depth": 20,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "criterion": "gini"
}

sklearn_decision_tree = DecisionTreeClassifier(
    max_depth=best_hyperparams["max_depth"],
    criterion=best_hyperparams["criterion"],
    min_samples_split=best_hyperparams["min_samples_split"],
    min_samples_leaf=best_hyperparams["min_samples_leaf"],
    random_state=random_seed
)
sklearn_decision_tree.fit(X_train, y_train)

y_train_preds = sklearn_decision_tree.predict(X_train)
y_pred = sklearn_decision_tree.predict(X_val)
y_pred_test = sklearn_decision_tree.predict(X_test)
val_accuracy = accuracy_score(y_val, y_pred)
train_accuracy = accuracy_score(y_train, y_train_preds)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Val Accuracy: {val_accuracy:.4f}")
print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Val Accuracy: 0.6733
Train Accuracy: 0.9753
Test Accuracy: 0.6667


### 2. Extract splits to JSON file

In [12]:
tree_dict = extract_tree_to_dict(sklearn_decision_tree)

with open(json_path, 'w') as f:
    json.dump(tree_dict, f, indent=2)

print(f"Tree saved to: {json_path}")

Tree saved to: ../models/decision_trees/fitted_decision_tree.json


## Verify Correct DT splits saved

### 1. Build saved DT from JSON  

In [13]:
custom_tree = DecisionTreeModel(json_path)

Tree loaded from: ../models/decision_trees/fitted_decision_tree.json


### 2. Generate Predictions from Extracted Splits and Verify Predictions Made

In [14]:
sklearn_predictions = sklearn_decision_tree.predict(X_val)
custom_predictions = custom_tree.predict(X_val.to_numpy())

matches = np.sum(sklearn_predictions == custom_predictions)
total = len(X_val)

print(f"Predictions match: {matches}/{total}")
print(f"\nsklearn accuracy: {accuracy_score(y_val, sklearn_predictions):.4f}")
print(f"Custom accuracy: {accuracy_score(y_val, custom_predictions):.4f}")

Predictions match: 150/150

sklearn accuracy: 0.6733
Custom accuracy: 0.6733
